# Candle Notebook: Minimal Candle + Image Save Test

This minimal notebook validates basic tensor creation and image saving without GUI. We also standardize:
- Working directory setup to the repo root
- Image store directory at `images_store`

Dependencies are declared in the next cell for reproducible builds.

In [ ]:
// Environment Setup - Path-Independent Dependencies
// Smart path resolution based on current working directory location

use std::path::Path;
let cwd = std::env::current_dir().unwrap();
let in_demos = cwd.to_string_lossy().contains("/demos");
let in_research = cwd.to_string_lossy().contains("/research/notebooks");

// Determine correct relative paths based on notebook location
let candle_core_path = if in_demos { "../../candle-core" } 
                      else if in_research { "../../../../candle-core" } 
                      else { "./candle-core" };

println!("📍 Working directory: {:?}", cwd);
println!("🔗 Using candle-core path: {}", candle_core_path);

// Dependencies with location-aware paths
:dep candle = { package = "candle-core", path = "../../../../candle-core" }
:dep image = "0.25"
:dep anyhow = "1.0"
:dep candle_notebooks = { package = "candle-notebooks", path = "../candle_notebooks" }

use candle::{Tensor, Device, DType};
use anyhow::Result;

In [ ]:
// Set working directory to repo root for consistent file operations
candle_notebooks::set_notebook_cwd().unwrap();
println!("📂 CWD set to repo root: {:?}", std::env::current_dir().unwrap());

In [ ]:
// Configure image store directory
use candle_notebooks as nb;
nb::set_image_store_rel_dir("images_store").unwrap();
println!("Image store directory: images_store");

In [ ]:
// Minimal candle test without GUI: build an RGB gradient and save with image crate.

fn save_image<P: AsRef<std::path::Path>>(img: &Tensor, p: P) -> Result<()> {
    let p = p.as_ref();
    let (c,h,w) = img.dims3()?; if c!=3 { anyhow::bail!("expected (3,H,W)"); }
    let img2 = img.permute((1,2,0))?.flatten_all()?;
    let pixels = img2.to_vec1::<u8>()?;
    let buffer: image::ImageBuffer<image::Rgb<u8>, Vec<u8>> = image::ImageBuffer::from_raw(w as u32,h as u32,pixels).ok_or_else(|| anyhow::anyhow!("raw to image"))?;
    buffer.save(p)?; Ok(())
}

// Create a simple RGB gradient in CHW layout as u8
let h = 140usize; let w = 200usize; let device = Device::Cpu;
let mut r = Vec::with_capacity(h*w); let mut g = Vec::with_capacity(h*w); let mut b = Vec::with_capacity(h*w);
for y in 0..h { for x in 0..w { r.push((x as f32 / w as f32 * 255.0) as f32); g.push((y as f32 / h as f32 * 255.0) as f32); b.push((((x+y) as f32)/(w+h) as f32 * 255.0) as f32); } }
let rt = Tensor::from_vec(r, (h,w), &device)?;
let gt = Tensor::from_vec(g, (h,w), &device)?;
let bt = Tensor::from_vec(b, (h,w), &device)?;
let rgb = Tensor::stack(&[&rt,&gt,&bt],0)?.to_dtype(DType::U8)?;
std::fs::create_dir_all("images_store")?;
save_image(&rgb, "images_store/test_gradient2.png")?;
println!("Saved images_store/test_gradient2.png");

Saved images_store/test_gradient.png
